# SVM Hyperparameter Trial Comparison

Run notebooks `00`–`04` first. This notebook loads their saved validation CSV files, combines the results, and creates a single comparison table. It does **not** use the test split.

## 1. Imports and Project Paths

In [ ]:
# Purpose: 1. Imports and Project Paths.
import os
from pathlib import Path
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    default_root = Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")
else:
    default_root = Path.cwd()
    if not (default_root / "Datasets").exists() and (default_root.parent / "Datasets").exists():
        default_root = default_root.parent

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", default_root))
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("TABLE_DIR:", TABLE_DIR)


## 2. Load Individual Trial Results

In [ ]:
# Purpose: 2. Load Individual Trial Results.
trial_files = {
    "baseline": "svm_baseline_validation.csv",
    "kernel": "svm_kernel_trial.csv",
    "C": "svm_c_trial.csv",
    "gamma": "svm_gamma_trial.csv",
    "class_weight": "svm_class_weight_trial.csv",
}

frames = []
missing = []
# Purpose: Iterates over this collection to build the next table, feature set, or experiment result
# Purpose: consistently.
for trial_name, filename in trial_files.items():
    path = TABLE_DIR / filename
    if not path.exists():
        missing.append(str(path))
        continue
    frame = pd.read_csv(path)
    frame.insert(0, "trial", trial_name)
    frames.append(frame)

if missing:
    print("Missing trial outputs:")
    # Purpose: Iterates over this collection to build the next table, feature set, or experiment result
    # Purpose: consistently.
    for path in missing:
        print(" -", path)

if not frames:
    raise FileNotFoundError("No SVM trial CSV outputs were found. Run notebooks 00-04 first.")

all_trials = pd.concat(frames, ignore_index=True, sort=False)
display(all_trials)


## 3. Best Validation Result from Each Trial

In [ ]:
# Purpose: 3. Best Validation Result from Each Trial.
best_per_trial = (
    all_trials.sort_values("f1", ascending=False)
    .groupby("trial", as_index=False)
    .first()
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

display(best_per_trial)

comparison_path = TABLE_DIR / "svm_individual_trial_comparison.csv"
# Purpose: Saves the generated artifacts so later notebooks, reports, or reruns can inspect the same outputs.
best_per_trial.to_csv(comparison_path, index=False)
print("Saved comparison:", comparison_path)


## 4. Important Methodology Note

The rows above come from **independent one-hyperparameter-at-a-time trials**. Do not automatically combine the best value from every row without checking the experimental plan: SVM hyperparameters can interact, especially `C` and `gamma`. Use the values required by your instructed trial sequence, then enter the final selected configuration into notebook `06_SVM_Final_Selected_Model.ipynb`.